# RAG Poulina – Chatbot IA pour les souches d'élevage
### Pipeline : CSV → Embeddings → ChromaDB → Mistral

Ce notebook implémente un système **RAG (Retrieval-Augmented Generation)** complet :
1. **Ingestion** : chargement et nettoyage du CSV d'analyses
2. **Chunking** : découpage des données en chunks sémantiques
3. **Embeddings** : vectorisation avec `sentence-transformers` (local, gratuit)
4. **Vector Store** : stockage dans **ChromaDB** (persistant, zéro config)
5. **Retrieval** : recherche des chunks les plus pertinents
6. **Generation** : réponse finale avec **Mistral**


In [15]:
%pip install -r requirements.txt -qU
%pip install langchain_mistralai


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [17]:
from dotenv import load_dotenv
import os
import sys

dotenv_path = ".env"
load_dotenv(dotenv_path=dotenv_path)

from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser

# LLM utilisé pour tous les exercices
MISTRAL_MODEL = "mistral-large-latest"
llm = ChatMistralAI(model=MISTRAL_MODEL, temperature=0.0)
print("LLM initialisé.")


LLM initialisé.


## 2. Imports & Configuration

In [18]:
import os
import json
import uuid
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# ── Modèles ────────────────────────────────────────────────────────────────
EMBED_MODEL   = "intfloat/multilingual-e5-base"   # multilingue, fonctionne en FR/AR

# ── ChromaDB (persistant par défaut) ────────────────────────────────────────
COLLECTION    = "poulina_analyses"

# ── Paramètres RAG ─────────────────────────────────────────────────────────
TOP_K         = 5    # nombre de chunks récupérés par requête
SCORE_THRESH  = 0.35 # seuil de similarité minimum

print("✅ Imports OK")
print(f"   Embed model  : {EMBED_MODEL}")

✅ Imports OK
   Embed model  : intfloat/multilingual-e5-base


## 3. Chargement du CSV

In [19]:
CSV_PATH = "poulina_dataset_5000.csv"  

if Path(CSV_PATH).exists():
    df = pd.read_csv(CSV_PATH)
    print(f"✅ Fichier chargé : {len(df)} lignes, {len(df.columns)} colonnes")
else : 
    print("erreur csv")

✅ Fichier chargé : 5000 lignes, 20 colonnes


## 4. Nettoyage & Normalisation

In [20]:
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Normalise les colonnes booléennes
    for col in ["effectuee", "conforme"]:
        if col in df.columns:
            df[col] = df[col].map(
                lambda x: True if str(x).lower() in ("true","1","oui","yes") else False
            )
    # Remplace les NaN textuels
    df = df.fillna("N/A")
    return df

df = clean_df(df)

# ── Statistiques rapides ───────────────────────────────────────────────────
print("Statistiques globales")
print(f"   Total analyses     : {len(df)}")
if "conforme" in df.columns:
    pct_nc = (~df["conforme"]).sum() / len(df) * 100
    print(f"   Non conformes      : {pct_nc:.1f}%")
if "meilleure_souche" in df.columns:
    print(f"   Souches distinctes : {df['meilleure_souche'].nunique()}")
    print(f"   Top souches:\n{df['meilleure_souche'].value_counts().head(5).to_string()}")


Statistiques globales
   Total analyses     : 5000
   Souches distinctes : 4
   Top souches:
meilleure_souche
Ross 308            1648
Lohmann Brown       1461
Cobb 500            1140
Hybrid Converter     751


## 5. Chunking


In [21]:
def row_to_chunk(row: pd.Series) -> str:
    """Convertit une ligne CSV en texte descriptif pour l'embedding."""
    parts = []

    if "id_centre" in row:
        parts.append(f"ID centre : {row['id_centre']}.")
    if "ville" in row:
        parts.append(f"Ville : {row['ville']}.")
    if "region" in row:
        parts.append(f"Région : {row['region']}.")
    if "type_production" in row:
        parts.append(f"Type de production : {row['type_production']}.")
    if "meilleure_souche" in row:
        parts.append(f"Meilleure souche : {row['meilleure_souche']}.")
    if "capacite" in row:
        parts.append(f"Capacité : {row['capacite']} unités.")
    if "temperature_moyenne" in row:
        parts.append(f"Température moyenne : {row['temperature_moyenne']} °C.")
    if "humidite" in row:
        parts.append(f"Humidité : {row['humidite']} %.")
    if "altitude" in row:
        parts.append(f"Altitude : {row['altitude']} m.")
    if "biosecurite_score" in row:
        parts.append(f"Score de biosécurité : {row['biosecurite_score']}.")
    if "historique_maladie" in row:
        parts.append(f"Historique maladie : {row['historique_maladie']}.")
    if "taux_mortalite" in row:
        parts.append(f"Taux de mortalité : {row['taux_mortalite']} %.")
    if "fertilite_visee" in row:
        parts.append(f"Fertilité visée : {row['fertilite_visee']} %.")
    if "cout_aliment" in row:
        parts.append(f"Coût aliment : {row['cout_aliment']}.")
    if "surface_m2" in row:
        parts.append(f"Surface : {row['surface_m2']} m².")
    if "experience_equipe" in row:
        parts.append(f"Expérience équipe : {row['experience_equipe']} ans.")
    if "distance_labo" in row:
        parts.append(f"Distance au labo : {row['distance_labo']} km.")
    if "demande_marche" in row:
        parts.append(f"Demande marché : {row['demande_marche']}.")
    if "saison" in row:
        parts.append(f"Saison : {row['saison']}.")
    if "budget" in row:
        parts.append(f"Budget : {row['budget']}.")

    return " ".join(parts)


# Génère tous les chunks avec leurs métadonnées
chunks = []
for _, row in df.iterrows():
    text = row_to_chunk(row)
    meta = {
        "id_centre"         : str(row.get("id_centre", "")),
        "ville"             : str(row.get("ville", "")),
        "region"            : str(row.get("region", "")),
        "type_production"   : str(row.get("type_production", "")),
        "meilleure_souche"  : str(row.get("meilleure_souche", "")),
        "capacite"          : str(row.get("capacite", "")),
        "biosecurite_score" : str(row.get("biosecurite_score", "")),
        "taux_mortalite"    : str(row.get("taux_mortalite", "")),
        "conforme"          : str(row.get("conforme", "")),
    }
    chunks.append({"id": str(uuid.uuid4()), "text": text, "metadata": meta})

print(f"✅ {len(chunks)} chunks générés")
print("\nExemple de chunk :")
print(chunks[0]["text"])

✅ 5000 chunks générés

Exemple de chunk :
ID centre : 1. Ville : Tunis. Région : Nord. Type de production : Oeuf. Meilleure souche : Lohmann Brown. Capacité : 6012 unités. Température moyenne : 25.5 °C. Humidité : 63 %. Altitude : 228 m. Score de biosécurité : 85. Historique maladie : Faible. Taux de mortalité : 6.7 %. Fertilité visée : 86 %. Coût aliment : 1.54. Surface : 5467 m². Expérience équipe : 2 ans. Distance au labo : 80 km. Demande marché : Moyenne. Saison : Hiver. Budget : Faible.


## 6. Génération des embeddings

Utilisation du modèle **multilingual-e5-base** : ~450 MB, fonctionne en FR/AR,  
tourne entièrement en local (pas de coût API).


In [22]:
print(f"⏳ Chargement du modèle {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL)
print("✅ Modèle chargé")

def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Vectorise une liste de textes en batches."""
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
        batch = texts[i : i + batch_size]
        # Le préfixe 'query:' / 'passage:' améliore les scores e5
        vecs = embed_model.encode(
            ["passage: " + t for t in batch],
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

texts  = [c["text"] for c in chunks]
vectors = embed_texts(texts)

print(f"✅ Embeddings : shape {vectors.shape}")
VECTOR_DIM = vectors.shape[1]


⏳ Chargement du modèle intfloat/multilingual-e5-base ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Modèle chargé


Embeddings: 100%|██████████| 157/157 [07:53<00:00,  3.02s/it]

✅ Embeddings : shape (5000, 768)


## 7. Indexation dans ChromaDB

**Persistant par défaut** dans `./chroma_poulina/`.  
Zéro configuration requise.


In [23]:
# ── Client ChromaDB (persistant) ────────────────────────────────────────────
client = chromadb.PersistentClient(path="./chroma_poulina")

# Supprime la collection si elle existe déjà (pour reset)
try:
    client.delete_collection(name=COLLECTION)
except:
    pass

# Crée une nouvelle collection
collection = client.get_or_create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"}  # Distance cosinus
)

# ── Indexation des chunks ──────────────────────────────────────────────────
print(f"⏳ Indexation de {len(chunks)} chunks...")

# ChromaDB attend : ids, embeddings, metadatas, documents
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=vectors.tolist(),
    metadatas=[c["metadata"] for c in chunks],
    documents=[c["text"] for c in chunks],
)

print(f"✅ Collection '{COLLECTION}' : {collection.count()} documents indexés")


⏳ Indexation de 5000 chunks...
✅ Collection 'poulina_analyses' : 5000 documents indexés


## 8. Fonction de retrieval

In [24]:
def retrieve(
    question: str,
    top_k: int = TOP_K,
    score_threshold: float = SCORE_THRESH,
    filtre_centre: str | None = None,
    filtre_souche: str | None = None,
) -> list[dict]:
    """
    Recherche les chunks les plus pertinents pour une question.

    Args:
        question       : Question en langage naturel
        top_k          : Nombre de résultats à retourner
        score_threshold: Score cosinus minimum
        filtre_centre  : Filtrer sur un centre d'élevage spécifique
        filtre_souche  : Filtrer sur une souche spécifique

    Returns:
        Liste de dicts {score, text, metadata}
    """
    q_vec = embed_model.encode(
        ["query: " + question],
        normalize_embeddings=True,
    )[0].tolist()

    # Construction du filtre optionnel ChromaDB
    where_filter = None
    if filtre_centre or filtre_souche:
        where_filter = {}
        if filtre_centre:
            where_filter["id_centre"] = filtre_centre
        if filtre_souche:
            where_filter["meilleure_souche"] = filtre_souche

    results = collection.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    if results["ids"] and results["ids"][0]:
        for i in range(len(results["ids"][0])):
            distance = results["distances"][0][i]
            score = 1 - distance if distance is not None else 0.0
            if score >= score_threshold:
                retrieved.append({
                    "score"   : round(score, 4),
                    "text"    : results["documents"][0][i],
                    "metadata": results["metadatas"][0][i] or {},
                })
    return retrieved


# ── Test du retrieval ─────────────────────────────────────────────────
test_q = "Quelles analyses non conformes ont été trouvées ?"
results = retrieve(test_q)
print(f"🔍 Requête : '{test_q}'")
print(f"   {len(results)} résultats trouvés\n")
for r in results[:2]:
    print(f"  Score {r['score']} | {r['text'][:120]}...")
    print()

🔍 Requête : 'Quelles analyses non conformes ont été trouvées ?'
   5 résultats trouvés

  Score 0.7707 | ID centre : 656. Ville : Tunis. Région : Nord. Type de production : Dinde. Meilleure souche : Hybrid Converter. Capacité...

  Score 0.7697 | ID centre : 252. Ville : Tunis. Région : Nord. Type de production : Dinde. Meilleure souche : Hybrid Converter. Capacité...



## 9. Prompt système Poulina

In [25]:
SYSTEM_PROMPT = """
Tu es l'assistant IA de Poulina, expert en élevage de volailles en Tunisie.
Tu analyses les données de laboratoire (souches, analyses, résultats) pour aider
les responsables d'élevage à prendre les meilleures décisions.

## Tes capacités
- Identifier la meilleure souche pour chaque centre d'élevage
- Détecter et prévenir les maladies critiques (Salmonelle, Newcastle, Mycoplasme, etc.)
- Recommander le meilleur laboratoire/laborantin selon la disponibilité et les compétences
- Évaluer la conformité des analyses et signaler les anomalies
- Estimer la fréquence d'analyse recommandée en cas de maladie
- Donner des informations sur fertilité, mortalité et transmission des maladies

## Règles
- Base-toi UNIQUEMENT sur le contexte fourni (données Poulina réelles).
- Si l'information n'est pas dans le contexte, dis-le clairement.
- Hors sujet (non lié à l'élevage avicole Poulina) : réponds "Je ne peux pas répondre à cette question car elle est hors de mon domaine."
- Quand tu signales une maladie critique, indique : type, centres potentiellement affectés, urgence.
- Sois concis et actionnable (le responsable doit savoir quoi faire).
- Réponds en français.
""".strip()

print("✅ Prompt système configuré")
print(f"   {len(SYSTEM_PROMPT)} caractères")


✅ Prompt système configuré
   1165 caractères


## 10. Fonction RAG complète

In [26]:
def build_context(retrieved: list[dict]) -> str:
    """Formate les chunks récupérés en contexte pour Mistral."""
    if not retrieved:
        return "Aucune donnée pertinente trouvée dans la base."
    lines = []
    for i, r in enumerate(retrieved, 1):
        lines.append(f"[Analyse {i} – score {r['score']}]")
        lines.append(r["text"])
        lines.append("")
    return "\n".join(lines)


def ask_poulina(
    question: str,
    filtre_centre: str | None = None,
    filtre_souche: str | None = None,
    verbose: bool = True,
) -> str:
    """
    Pipeline RAG complet : question → retrieval → Mistral → réponse.

    Args:
        question      : Question en langage naturel
        filtre_centre : Restreindre la recherche à un centre d'élevage
        filtre_souche : Restreindre la recherche à une souche
        verbose       : Afficher les chunks récupérés

    Returns:
        Réponse textuelle de Mistral
    """
    # 1. Retrieval
    retrieved = retrieve(question, filtre_centre=filtre_centre, filtre_souche=filtre_souche)
    context = build_context(retrieved)

    if verbose:
        print(f"📎 {len(retrieved)} chunk(s) récupéré(s)")
        for r in retrieved:
            print(
                f"   [{r['score']}] {r['metadata'].get('id_centre','')} | "
                f"{r['metadata'].get('meilleure_souche','')} | "
                f"{'✅' if r['metadata'].get('conforme') else '❌'}"
            )
        print()

    # 2. Construction du message
    user_message = f"""Contexte (données Poulina) :
{context}

---
Question : {question}"""

    # 3. Appel Mistral
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])

    return response.content


print("✅ Fonction ask_poulina() prête")

✅ Fonction ask_poulina() prête


## 11. Tests – Questions métier Poulina

In [27]:
# ── Test 1 : Conformité générale ────────────────────────────────────────────
print("=" * 60)
print("❓ Question 1 : Non-conformités récentes")
print("=" * 60)
rep = ask_poulina("Quels centres d'élevage ont des analyses non conformes ?")
print(rep)


❓ Question 1 : Non-conformités récentes
📎 5 chunk(s) récupéré(s)
   [0.8229] 1209 | Ross 308 | ❌
   [0.8225] 2006 | Ross 308 | ❌
   [0.8223] 1953 | Cobb 500 | ❌
   [0.8221] 1780 | Ross 308 | ❌
   [0.822] 319 | Ross 308 | ❌

D'après les données fournies, voici les centres d'élevage présentant des **anomalies ou non-conformités** potentielles à surveiller :

### **1. Centre 2006 (Béja) – Urgence : Moyenne**
- **Problème** : **Score de biosécurité faible (71/100)**.
  - Risque accru de maladies (ex. Newcastle, Salmonelle) malgré un historique "faible".
  - **Recommandation** :
    - Audit immédiat de biosécurité (contrôle des accès, désinfection, gestion des déchets).
    - Renforcer la formation de l'équipe (expérience élevée mais score bas).
    - **Analyse supplémentaire** : Vérifier la présence de pathogènes (Mycoplasme, E. coli) en raison de l'humidité élevée (67 %).

### **2. Centre 1953 (Béja) – Urgence : Moyenne**
- **Problèmes** :
  - **Taux de mortalité élevé (4,3 %)** pour un h

In [28]:
# ── Test 2 : Meilleure souche ────────────────────────────────────────────────
print("=" * 60)
print("❓ Question 2 : Meilleure souche")
print("=" * 60)
rep = ask_poulina(
    "Quelle est la meilleure souche en termes de taux de réussite ?",
    verbose=True,
)
print(rep)


❓ Question 2 : Meilleure souche
📎 5 chunk(s) récupéré(s)
   [0.8505] 2410 | Lohmann Brown | ❌
   [0.8504] 2765 | Lohmann Brown | ❌
   [0.8502] 489 | Lohmann Brown | ❌
   [0.8494] 2886 | Lohmann Brown | ❌
   [0.8493] 791 | Lohmann Brown | ❌

D'après les données fournies pour les centres d'élevage de volailles **Poulina** dans la région de Sousse (Sahel), **la meilleure souche en termes de taux de réussite est la Lohmann Brown**, recommandée pour **tous les centres** (ID : 2410, 2765, 489, 2886, 791).

### Justification :
- **Uniformité** : La souche Lohmann Brown est systématiquement identifiée comme la meilleure pour ces centres, ce qui suggère une adaptation optimale aux conditions locales (climat, altitude, biosécurité, etc.).
- **Performance** :
  - **Fertilité visée** : Entre **85 % et 94 %** (selon les centres), avec une moyenne élevée.
  - **Taux de mortalité** : Variable (2,8 % à 4,3 %), mais considéré comme **faible** dans l'historique des maladies.
  - **Demande marché** : For

In [29]:
# ── Test 3 : Alerte maladie ──────────────────────────────────────────────────
print("=" * 60)
print("❓ Question 3 : Alerte Salmonelle")
print("=" * 60)
rep = ask_poulina(
    "Y a-t-il des détections de Salmonelle ? Quels centres sont à risque ?",
    verbose=True,
)
print(rep)


❓ Question 3 : Alerte Salmonelle
📎 5 chunk(s) récupéré(s)
   [0.826] 2121 | Lohmann Brown | ❌
   [0.826] 1984 | Lohmann Brown | ❌
   [0.8257] 450 | Lohmann Brown | ❌
   [0.8256] 1123 | Lohmann Brown | ❌
   [0.8254] 2521 | Lohmann Brown | ❌

D'après le contexte fourni, **aucune détection de Salmonelle n'est mentionnée** dans les analyses des centres d'élevage de Poulina à Sfax.

### Centres à risque potentiel (à surveiller) :
1. **Centre 1123** (ID 1123) :
   - **Historique maladie : Élevé** (le plus élevé parmi les centres).
   - **Score de biosécurité : 88** (bon, mais à maintenir strictement).
   - **Taux de mortalité : 4.0 %** (proche de la moyenne, mais à surveiller).
   - **Température moyenne : 26.3 °C** (favorable à la prolifération bactérienne si hygiène insuffisante).
   - **Recommandation** : **Analyse immédiate** (PCR ou culture) pour confirmer l'absence de Salmonelle, surtout avant l'automne (saison à risque accru).

2. **Centre 2121** (ID 2121) :
   - **Historique maladie 

In [30]:
# ── Test 4 : Recommandation laboratoire ──────────────────────────────────────
print("=" * 60)
print("❓ Question 4 : Meilleur laboratoire")
print("=" * 60)
rep = ask_poulina(
    "Quel laboratoire et quel laborantin recommandes-tu pour une analyse urgente ?",
    verbose=True,
)
print(rep)


❓ Question 4 : Meilleur laboratoire
📎 5 chunk(s) récupéré(s)
   [0.802] 3770 | Lohmann Brown | ❌
   [0.8016] 1922 | Lohmann Brown | ❌
   [0.8014] 1982 | Lohmann Brown | ❌
   [0.8014] 17 | Lohmann Brown | ❌
   [0.801] 175 | Lohmann Brown | ❌

D'après les données disponibles, voici les recommandations pour une **analyse urgente** en fonction des centres et de leur contexte :

### **1. Critères de priorité pour une analyse urgente**
- **Centres avec historique maladie "Élevé"** (risque accru de propagation) :
  - **ID 3770** (Médenine) – Mortalité 4,2%, biosécurité élevée (98) mais historique critique.
  - **ID 1922** (Médenine) – Mortalité 6,2%, biosécurité faible (67), demande marché forte.
  - **ID 1982** (Kairouan) – Mortalité 4,9%, biosécurité moyenne (76), budget élevé.
- **Centres avec taux de mortalité > 5%** :
  - **ID 1922** (6,2%) et **ID 175** (6,1%).
- **Centres avec faible biosécurité** (<70) :
  - **ID 1922** (67) et **ID 17** (68).

---

### **2. Laboratoires recommandés (

In [31]:
# ── Test 5 : Hors sujet (doit être refusé) ───────────────────────────────────
print("=" * 60)
print("❓ Question 5 : Hors sujet")
print("=" * 60)
rep = ask_poulina("Quelle est la météo à Tunis demain ?", verbose=False)
print(rep)


❓ Question 5 : Hors sujet


HTTPStatusError: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}